In [1]:
# Trazabilidad F1 - Importacion de librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
print('Librerias importadas correctamente')

Librerias importadas correctamente


In [2]:
# Trazabilidad F2 - Carga del dataset de inferencia 2025
inferencia_df = pd.read_csv('..//data//raw//inferencia//ventas_2025_inferencia.csv')
inferencia_df['fecha'] = pd.to_datetime(inferencia_df['fecha'])

print(f'Registros cargados: {len(inferencia_df)}')
print(f'Columnas: {list(inferencia_df.columns)}')
print(f'Rango de fechas: {inferencia_df["fecha"].min()} a {inferencia_df["fecha"].max()}')
print(f'Productos: {inferencia_df["nombre"].nunique()}')
inferencia_df.head()

Registros cargados: 888
Columnas: ['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria', 'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta', 'ingresos', 'Amazon', 'Decathlon', 'Deporvillage']
Rango de fechas: 2025-10-25 00:00:00 a 2025-11-30 00:00:00
Productos: 24


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


In [3]:
# Trazabilidad F3 - Verificacion de columnas de competencia
# Los datos de inferencia ya incluyen Amazon, Decathlon, Deporvillage en el mismo CSV
columnas_competencia = ['Amazon', 'Decathlon', 'Deporvillage']
tiene_competencia = all(col in inferencia_df.columns for col in columnas_competencia)

print(f'Columnas de competencia presentes: {tiene_competencia}')
print(f'Columnas actuales: {list(inferencia_df.columns)}')

Columnas de competencia presentes: True
Columnas actuales: ['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria', 'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta', 'ingresos', 'Amazon', 'Decathlon', 'Deporvillage']


In [4]:
# Trazabilidad F4 - Calculo de precio_competencia y ratio_precio
inferencia_df['precio_competencia'] = inferencia_df[['Amazon', 'Decathlon', 'Deporvillage']].mean(axis=1)
inferencia_df['ratio_precio'] = np.where(
    inferencia_df['precio_competencia'] != 0,
    inferencia_df['precio_base'] / inferencia_df['precio_competencia'],
    np.nan
)

# Eliminar columnas de competidores individuales
inferencia_df = inferencia_df.drop(columns=['Amazon', 'Decathlon', 'Deporvillage'])

print(f'Despues de calcular competencia: {inferencia_df.shape}')
print(f'Columnas: {list(inferencia_df.columns)}')

Despues de calcular competencia: (888, 12)
Columnas: ['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria', 'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta', 'ingresos', 'precio_competencia', 'ratio_precio']


In [5]:
# Trazabilidad F5 - One-Hot Encoding y alineacion de columnas con df.csv
# Crear copias de variables categoricas con sufijo _h
inferencia_df['nombre_h'] = inferencia_df['nombre']
inferencia_df['categoria_h'] = inferencia_df['categoria']
inferencia_df['subcategoria_h'] = inferencia_df['subcategoria']

# One-hot encoding
inferencia_df = pd.get_dummies(inferencia_df, columns=['nombre_h', 'categoria_h', 'subcategoria_h'], dtype=int)

# Cargar columnas de referencia de df.csv
df_ref = pd.read_csv('..//data//processed//df.csv', nrows=0)
columnas_ref = list(df_ref.columns)

# Anadir columnas faltantes como 0
for col in columnas_ref:
    if col not in inferencia_df.columns:
        inferencia_df[col] = 0

# Eliminar columnas extras que no estan en df.csv
columnas_extras = [col for col in inferencia_df.columns if col not in columnas_ref]
inferencia_df = inferencia_df.drop(columns=columnas_extras)

# Reordenar columnas igual que df.csv
inferencia_df = inferencia_df[columnas_ref]

print(f'Despues de alineacion: {inferencia_df.shape}')
print(f'Columnas iguales a df.csv: {list(inferencia_df.columns) == columnas_ref}')

Despues de alineacion: (888, 56)
Columnas iguales a df.csv: True


In [6]:
# Trazabilidad F6 - Filtrado: eliminar octubre, solo noviembre + guardado CSV
registros_antes = len(inferencia_df)

# Eliminar registros de octubre (dia_mes <= 31 de octubre)
inferencia_df = inferencia_df[inferencia_df['fecha'].dt.month == 11].reset_index(drop=True)

registros_despues = len(inferencia_df)
eliminados = registros_antes - registros_despues

print(f'Registros antes del filtrado: {registros_antes}')
print(f'Registros eliminados (octubre): {eliminados}')
print(f'Registros despues del filtrado: {registros_despues}')
print(f'Shape final: {inferencia_df.shape}')
print(f'Rango de fechas: {inferencia_df["fecha"].min()} a {inferencia_df["fecha"].max()}')

# Guardar CSV
os.makedirs('..//data//processed', exist_ok=True)
inferencia_df.to_csv('..//data//processed//inferencia_df_transformado.csv', index=False)
print(f'\nGuardado en data/processed/inferencia_df_transformado.csv')

Registros antes del filtrado: 888
Registros eliminados (octubre): 168
Registros despues del filtrado: 720
Shape final: (720, 56)
Rango de fechas: 2025-11-01 00:00:00 a 2025-11-30 00:00:00

Guardado en data/processed/inferencia_df_transformado.csv


In [11]:
inferencia_df.head()

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,subcategoria_h_Esterilla Yoga,subcategoria_h_Mancuernas Ajustables,subcategoria_h_Mochila Trekking,subcategoria_h_Pesa Rusa,subcategoria_h_Pesas Casa,subcategoria_h_Rodillera Yoga,subcategoria_h_Ropa Montaña,subcategoria_h_Ropa Running,subcategoria_h_Zapatillas Running,subcategoria_h_Zapatillas Trail
0,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.00,NaN,...,0,0,0,0,0,0,0,0,1,0
1,2025-11-01,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,NaN,135.00,NaN,...,0,0,0,0,0,0,0,0,1,0
2,2025-11-01,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,NaN,86.39,NaN,...,0,0,0,0,0,0,0,0,1,0
3,2025-11-01,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,NaN,74.09,NaN,...,0,0,0,0,0,0,0,0,1,0
4,2025-11-01,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,NaN,34.76,NaN,...,0,0,0,0,0,0,0,1,0,0
